# Rap-Snacks ColabFold Validation v2

Folds four sequence buckets through AlphaFold2 (ColabFold) to validate:

| Bucket | n | Label |
|--------|---|-------|
| Concordance originals | 12 | `concordance` |
| Native-alanine originals | 12 | `native_ala` |
| Free-design MPNN (top-5 diverse/bar) | 60 | `free_design` |
| Scrambled controls (5/bar) | 60 | `scrambled` |

**Expected result:** `scrambled` ≪ `concordance` ≈ `native_ala` < `free_design`

**Runtime:** ~10–15 min on A100 (single_sequence mode, 1 model, 1 recycle)

---
## Cell 1 — Install

**Run this cell, then Runtime → Restart runtime, then continue from Cell 2.**

Do NOT skip the restart — JAX must be loaded fresh after install.

In [ ]:
import subprocess, sys, os

# Step 1: colabfold WITHOUT jax (alphafold-minus-jax skips jax/jaxlib install)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts',
    'colabfold[alphafold-minus-jax]'
], check=True)

# Step 2: pin jax + jaxlib + cuda plugin to matching versions
# --force-reinstall ensures colabfold's deps don't leave a newer plugin behind
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', '--force-reinstall',
    'jax==0.4.25',
    'jaxlib==0.4.25+cuda12.cudnn89',
    'jax-cuda12-plugin==0.4.25',
    '-f', 'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html'
], check=True)

print('Install complete — Runtime → Restart runtime, then run Cell 2 onwards.')

---
## Cell 2 — Mount Drive + Paths

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive/rap_snacks')
DRIVE_IN   = DRIVE_ROOT / 'inputs'
DRIVE_RES  = DRIVE_ROOT / 'results'
DRIVE_FIGS = DRIVE_RES / 'figures'
CONTENT    = Path('/content/scratch')

for p in [DRIVE_IN, DRIVE_RES, DRIVE_FIGS, CONTENT]:
    p.mkdir(parents=True, exist_ok=True)

# Expected Drive inputs:
#   inputs/data/aggregated_lines_v2_enriched.csv
#   inputs/data/phase2_candidates.csv
#   inputs/outputs/proteinmpnn_free/filtered_results.csv
#   inputs/outputs/scrambled/scrambled_results.csv

DATA      = DRIVE_IN / 'data'
MPNN_FREE = DRIVE_IN / 'outputs' / 'proteinmpnn_free' / 'filtered_results.csv'
SCRAMBLED = DRIVE_IN / 'outputs' / 'scrambled' / 'scrambled_results.csv'

for p in [DATA / 'aggregated_lines_v2_enriched.csv',
          DATA / 'phase2_candidates.csv',
          MPNN_FREE, SCRAMBLED]:
    status = '✅' if p.exists() else '❌ MISSING'
    print(f'{status}  {p.relative_to(DRIVE_ROOT)}')

print('\nPaths ready.')

---
## Cell 3 — Config

In [ ]:
import json, random, subprocess, sys, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TOP_N        = 12
N_SCRAMBLES  = 5      # scrambled seqs per bar
N_FREE       = 5      # free_design seqs per bar (most diverse)
SEED         = 42

# ColabFold
NUM_RECYCLE  = 1
MODEL_TYPE   = 'alphafold2_ptm'
NUM_MODELS   = 1
CHUNK_SIZE   = 150

# Scratch paths
CF_OUT       = CONTENT / 'cf_output'
FASTA_PATH   = CONTENT / 'validation_seqs.fasta'

print('Config loaded.')

---
## Cell 4 — Helpers

In [ ]:
def parse_colabfold_scores(scores_dir):
    """
    Parse ColabFold output. Returns {name: {'plddt': float 0-1, 'ptm': float}}.
    ColabFold stores pLDDT as 0-100 in JSON — divide by 100.
    """
    results = {}
    for p in sorted(Path(scores_dir).glob('*_scores_rank_001*.json')):
        name = p.name.split('_scores_rank_001')[0]
        data = json.loads(p.read_text())
        results[name] = {
            'plddt': float(np.mean(data['plddt'])) / 100.0,
            'ptm':   data.get('ptm'),
        }
    return results


def run_colabfold(fasta_path, out_dir, msa_mode='single_sequence', chunk_size=CHUNK_SIZE):
    import tempfile
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    records, header, seq = [], None, []
    for line in Path(fasta_path).read_text().splitlines():
        if line.startswith('>'):
            if header: records.append((header, ''.join(seq)))
            header, seq = line[1:].strip(), []
        else:
            seq.append(line.strip())
    if header: records.append((header, ''.join(seq)))

    total  = len(records)
    chunks = [records[i:i+chunk_size] for i in range(0, total, chunk_size)]
    print(f'{total} sequences → {len(chunks)} chunk(s) of ≤{chunk_size}')

    for ci, chunk in enumerate(chunks):
        print(f'  Chunk {ci+1}/{len(chunks)} ({len(chunk)} seqs) ...', flush=True)
        with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False) as tmp:
            for h, s in chunk: tmp.write(f'>{h}\n{s}\n')
            tmp_path = tmp.name
        cmd = [
            'colabfold_batch', tmp_path, str(out_dir),
            '--num-recycle', str(NUM_RECYCLE),
            '--model-type',  MODEL_TYPE,
            '--num-models',  str(NUM_MODELS),
            '--msa-mode',    msa_mode,
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'STDERR (last 3000 chars):\n{r.stderr[-3000:]}')
            raise RuntimeError(f'ColabFold failed on chunk {ci+1} (exit {r.returncode})')
        Path(tmp_path).unlink(missing_ok=True)
        print(f'    chunk {ci+1} done.')

    return parse_colabfold_scores(out_dir)


def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))


def greedy_diverse(seqs, n=5):
    """Greedy max-min: pick n sequences maximally spread in sequence space."""
    selected = [0]
    while len(selected) < min(n, len(seqs)):
        best, best_dist = -1, -1
        for i in range(len(seqs)):
            if i in selected: continue
            min_dist = min(hamming(seqs[i], seqs[j]) for j in selected)
            if min_dist > best_dist:
                best, best_dist = i, min_dist
        selected.append(best)
    return selected


print('Helpers loaded.')

---
## Cell 5 — Build Validation FASTA

Assembles all four buckets into a single FASTA for ColabFold.

In [ ]:
VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')

enriched   = pd.read_csv(DATA / 'aggregated_lines_v2_enriched.csv')
candidates = pd.read_csv(DATA / 'phase2_candidates.csv').head(TOP_N)
bar_ids    = list(candidates['bar_id'])
sub        = enriched[enriched['bar_id'].isin(bar_ids)].set_index('bar_id')

free_df    = pd.read_csv(MPNN_FREE)
sc_df      = pd.read_csv(SCRAMBLED)

rng        = random.Random(SEED)
records    = []   # (cf_name, sequence, bar_id, bucket, detail)

for bar_id in bar_ids:

    # ── concordance ──────────────────────────────────────────────────────────
    conc_seq = sub.loc[bar_id, 'fasta_seq_concordance']
    if pd.notna(conc_seq) and set(conc_seq.upper()) <= VALID_AA:
        records.append((f'{bar_id}_concordance', conc_seq, bar_id, 'concordance', ''))

    # ── native_ala ────────────────────────────────────────────────────────────
    na_seq = sub.loc[bar_id, 'fasta_seq_native_alanine']
    if pd.notna(na_seq) and set(na_seq.upper()) <= VALID_AA:
        records.append((f'{bar_id}_native_ala', na_seq, bar_id, 'native_ala', ''))

    # ── free_design — top-N most diverse ─────────────────────────────────────
    grp = free_df[free_df['bar_id'] == bar_id].drop_duplicates('sequence').reset_index(drop=True)
    if len(grp):
        seqs_list = list(grp['sequence'])
        idx = greedy_diverse(seqs_list, n=N_FREE)
        for rank, i in enumerate(idx):
            row = grp.iloc[i]
            records.append((
                f'{bar_id}_free_{rank:02d}',
                row['sequence'], bar_id, 'free_design', f'rank{rank}_plddt{row["esm_plddt"]:.3f}'
            ))
    else:
        print(f'  [WARN] No free_design sequences for {bar_id}')

    # ── scrambled — 5 per bar (concordance shuffles) ──────────────────────────
    sc_bar = sc_df[(sc_df['bar_id'] == bar_id) &
                   (sc_df['source'] == 'scrambled_concordance') &
                   (sc_df['plddt'].notna())].head(N_SCRAMBLES)
    for j, (_, sc_row) in enumerate(sc_bar.iterrows()):
        seq = sc_row['seq']
        if pd.notna(seq) and set(seq.upper()) <= VALID_AA:
            records.append((f'{bar_id}_scrambled_{j:02d}', seq, bar_id, 'scrambled', ''))

# Write FASTA
with open(FASTA_PATH, 'w') as f:
    for name, seq, *_ in records:
        f.write(f'>{name}\n{seq}\n')

# Summary
meta = pd.DataFrame(records, columns=['cf_name', 'sequence', 'bar_id', 'bucket', 'detail'])
print(f'Total sequences: {len(meta)}')
print(meta.groupby('bucket').size().rename('count').to_string())
print(f'\nFASTA written → {FASTA_PATH}')

---
## Cell 6 — Run ColabFold

~10–15 min on A100. `single_sequence` mode — no MSA lookup (correct for designed sequences).

In [ ]:
scores = run_colabfold(FASTA_PATH, CF_OUT, msa_mode='single_sequence')
print(f'\nScored {len(scores)} sequences.')

---
## Cell 7 — Merge Results + Save to Drive

In [ ]:
meta['cf_plddt'] = meta['cf_name'].map(lambda n: scores.get(n, {}).get('plddt'))
meta['cf_ptm']   = meta['cf_name'].map(lambda n: scores.get(n, {}).get('ptm'))

results_path = CONTENT / 'validation_results.csv'
meta.to_csv(results_path, index=False)

# Save to Drive
shutil.copy2(results_path, DRIVE_RES / 'validation_results_v2.csv')
print(f'Saved → Drive: results/validation_results_v2.csv')

# Summary
print('\n--- ColabFold pLDDT by bucket ---')
for bucket, grp in meta.groupby('bucket'):
    vals = grp['cf_plddt'].dropna()
    print(f'  {bucket:15s}  n={len(vals):3d}  mean={vals.mean():.3f}  '
          f'sd={vals.std():.3f}  min={vals.min():.3f}  max={vals.max():.3f}')

---
## Cell 8 — Figures

In [ ]:
DARK_BG = '#0e0e0e'
COLORS  = {
    'concordance': '#00d4ff',
    'native_ala':  '#ff9500',
    'free_design': '#a855f7',
    'scrambled':   '#444455',
}
ORDER   = ['scrambled', 'concordance', 'native_ala', 'free_design']

# ── Fig A: strip plot per bar ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

bar_ids_sorted = sorted(meta['bar_id'].unique())
n_bars  = len(bar_ids_sorted)
n_buck  = len(ORDER)
width   = 0.18
offsets = np.linspace(-(n_buck-1)*width/2, (n_buck-1)*width/2, n_buck)

plotted = set()
for bi, bar_id in enumerate(bar_ids_sorted):
    for oi, bucket in enumerate(ORDER):
        vals = meta[(meta['bar_id']==bar_id) & (meta['bucket']==bucket)]['cf_plddt'].dropna()
        if vals.empty: continue
        x = bi + offsets[oi]
        label = bucket if bucket not in plotted else ''
        ax.scatter([x]*len(vals), vals, color=COLORS[bucket], s=30,
                   alpha=0.8, label=label, zorder=3)
        ax.plot([x-0.05, x+0.05], [vals.mean(), vals.mean()],
                color=COLORS[bucket], linewidth=2, zorder=4)
        plotted.add(bucket)

ax.axhline(0.5, color='white', linestyle='--', linewidth=0.8, alpha=0.4, label='pLDDT=0.5')
ax.set_xticks(range(n_bars))
ax.set_xticklabels(bar_ids_sorted, rotation=45, ha='right', fontsize=8, color='white')
ax.set_ylabel('ColabFold pLDDT (0–1)', color='white')
ax.set_ylim(0, 1)
ax.set_title('ColabFold pLDDT: free_design vs concordance vs native_ala vs scrambled',
             color='white', fontsize=12)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#333333')
handles, labels = ax.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labels):
    if l and l not in seen: seen[l] = h
ax.legend(seen.values(), seen.keys(), facecolor='#1a1a1a',
          labelcolor='white', framealpha=0.8, fontsize=9)

plt.tight_layout()
fig_path_a = CONTENT / 'fig_cf_per_bar.png'
plt.savefig(fig_path_a, dpi=150, facecolor=fig.get_facecolor())
shutil.copy2(fig_path_a, DRIVE_FIGS / 'fig_cf_per_bar.png')
plt.show()
print('Fig A saved.')

# ── Fig B: violin / box per bucket ───────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 5), facecolor=DARK_BG)
ax2.set_facecolor(DARK_BG)

data_by_bucket = [meta[meta['bucket']==b]['cf_plddt'].dropna().values for b in ORDER]
parts = ax2.violinplot(data_by_bucket, positions=range(len(ORDER)),
                       showmedians=True, showextrema=False)
for i, (pc, bucket) in enumerate(zip(parts['bodies'], ORDER)):
    pc.set_facecolor(COLORS[bucket])
    pc.set_alpha(0.7)
parts['cmedians'].set_color('white')

ax2.set_xticks(range(len(ORDER)))
ax2.set_xticklabels(ORDER, color='white', fontsize=10)
ax2.set_ylabel('ColabFold pLDDT (0–1)', color='white')
ax2.set_ylim(0, 1)
ax2.set_title('pLDDT distribution by bucket', color='white', fontsize=12)
ax2.tick_params(colors='white')
for spine in ax2.spines.values(): spine.set_edgecolor('#333333')
ax2.axhline(0.5, color='white', linestyle='--', linewidth=0.8, alpha=0.4)

plt.tight_layout()
fig_path_b = CONTENT / 'fig_cf_violin.png'
plt.savefig(fig_path_b, dpi=150, facecolor=fig2.get_facecolor())
shutil.copy2(fig_path_b, DRIVE_FIGS / 'fig_cf_violin.png')
plt.show()
print('Fig B saved.')